In [9]:
import numpy as np
import matplotlib.pyplot as plt
import math
import cmath

In [10]:
def parallel_resistance(r1, r2):
    return 1/((1/r1)+(1/r2))

In [1]:
# System Parameters

accumulator_series_elements = 10*14
cell_max_voltage = 4.2 #volts
cell_min_voltage = 2.5 #volts

min_pack_voltage = cell_min_voltage*accumulator_series_elements
print(min_pack_voltage)
max_pack_voltage = cell_max_voltage*accumulator_series_elements
mid_pack_voltage = (min_pack_voltage + max_pack_voltage)/2

pmax = 850 #Watts
vout = 320 #volts
iout = pmax/vout
print(iout)

i_in_min = pmax/max_pack_voltage
i_in_max = pmax/min_pack_voltage

acceptable_dvout = 5 #volts

fsw = 50*10**3 #Hz

acceptable_dv_in = 10


350.0
2.65625


In [33]:
# Duty Cycle Calculations

max_duty_cycle = vout/min_pack_voltage
min_duty_cycle = vout/max_pack_voltage

print(max_duty_cycle)
print(min_duty_cycle)

0.9142857142857143
0.54421768707483


In [34]:
# FET Calculations
# https://www.ti.com/lit/an/slyt664/slyt664.pdf?ts=1745717291353&ref_url=https%253A%252F%252Fwww.google.com%252F

ton_top = (max_duty_cycle)/fsw # worst case
print(ton_top)
ton_bot = (1-min_duty_cycle)/fsw # worst case
print(ton_bot)

switching_fet_rdson = 0.025 # Ohms #TODO placeholder value
switching_fet_qg = 40*10**-9 # C #TODO placeholder value
recirculation_diode_vf = 0.85 # V
recirculaton_diode_c_junc = 45*10**-12 #F
recirculation_diode_qrr = recirculaton_diode_c_junc*max_pack_voltage
print(recirculation_diode_qrr)
tr = 13.5*10**-9
tf = 16*10**-9

switching_fet_conduction_losses_top = ton_top*switching_fet_rdson*i_in_max**2
recirculation_diode_conduction_losses = 1
#TODO consider affect of ripple in current

gate_driver_source_current = 4 #A
#TODO consider asymmetrical drive currents

switching_fet_switching_losses = max_pack_voltage*i_in_max*fsw*(switching_fet_qg/gate_driver_source_current)
recirculation_diode_recovery_losses = fsw*max_pack_voltage*recirculation_diode_qrr/3
print(recirculation_diode_recovery_losses)

switching_fet_total_losses_top = switching_fet_conduction_losses_top + switching_fet_switching_losses
recirculation_diode_total_losses = recirculation_diode_recovery_losses + recirculation_diode_conduction_losses
print(recirculation_diode_total_losses)

print(switching_fet_total_losses_top) #Watts

switching_fraction = (tr)*fsw
print(switching_fraction)
print(1/fsw)

avg_gate_drive_current = gate_driver_source_current*switching_fraction
print(avg_gate_drive_current)

1.8285714285714284e-05
9.1156462585034e-06
2.646e-08
0.259308
1.2593079999999999
0.7140026962099125
0.000675
2e-05
0.0027


In [35]:
# Gate Driver Calculations
bootstrap_charge_voltage = 15
bootstrap_diode_vf = 1.3
bootstrap_resistance = 2.2


I_boot_peak = (bootstrap_charge_voltage-bootstrap_diode_vf)/bootstrap_resistance
print(I_boot_peak)

switching_fet_qg = 40*10**-9 #C #TODO placeholder value
gate_driver_quiescent_current = 0.0025 #A
acceptable_chargePump_ripple = 0.5 #V

q_total_chargePump = switching_fet_qg + gate_driver_quiescent_current/fsw
c_boot_min = q_total_chargePump/acceptable_chargePump_ripple
print(c_boot_min*10**9) #nF


6.227272727272727
180.0


In [36]:
# Equivalent Capacitance Function Definition
# https://www.power-mag.com/pdf/feature_pdf/1387888355_Murata_Feature_Layout_1.pdf

#Returned datastructure
class EquivalentCapacitance:
    def __init__(self, capacitances, fsw, ripple_current, equivalent_capacitance, equivalent_esr, ripple_voltage):
        self.capacitances = capacitances # Will be of the form [qty, capacitance (F), esr (Ohm), ripple current (A)] where the 4th column will only be added if a total ripple current was provided
        self.fsw = fsw
        self.ripple_current = ripple_current
        self.equivalent_capacitance = equivalent_capacitance
        self.equivalent_esr = equivalent_esr
        self.ripple_voltage = ripple_voltage

    def print(self):
        print(f"Capacitance Matrix: {self.capacitances}")
        print(f"Switching Frequency: {self.fsw} Hz")
        print(f"Ripple Current: {self.ripple_current} A")
        print(f"Equivalent Capacitance: {self.equivalent_capacitance} F")
        print(f"Equivalent ESR: {self.equivalent_esr} Ohms")
        print(f"Ripple Voltage: {self.ripple_voltage} V")

def equiv_capacitance(capacitances, fsw, current = 0):
    # capacitances is a 3 column, n row matrix with rows of the form [qty, capacitance (uF), esr (mOhm)]
    # fsw is the fundamental switching frequency (kHz)
    # current is optional. If provided, the return will include the ripple voltage and current per capacitor
    
    cs = np.array(capacitances)
    cs = cs.astype(float)
    xs = 1/(2*3.1416*fsw*cs[:, 1])
    
    cp = cs.copy()
    cp[:, 1] = cs[:, 1]/(1 + (cs[:, 2]/xs)**2)
    cp[:, 2] = (cs[:, 2]**2 + xs**2)/cs[:, 2]
    
    cpe = np.sum(cp[:, 0]*cp[:, 1])
    rpe = 1/np.sum(cp[:, 0]/cp[:, 2])

    xpe = 1/(2*3.1416*fsw*cpe)
    cse = cpe*(1 + (xpe/rpe)**2)
    rse = rpe/(1 + (rpe/xpe)**2)
    xse = 1/(2*3.1416*fsw*cse)

    dv = np.nan
    if(current != 0):
        dv = current*math.sqrt(rse**2 + xse**2)
        di = dv/np.sqrt(cs[:, 2]**2 + xs**2)
        cs = np.hstack((cs, di.reshape(-1, 1)))
    return EquivalentCapacitance(cs, fsw, current if current > 0 else np.nan, cse, rse, dv)


In [37]:
# Filter Calculations
# https://www.ti.com/lit/an/slva477b/slva477b.pdf?ts=1745747396768

acceptable_inductor_ripple_current = 0.2*iout #Amps
print(acceptable_inductor_ripple_current)
calculated_inductance = vout*(max_pack_voltage-vout)/(acceptable_inductor_ripple_current*fsw*max_pack_voltage)
print(calculated_inductance*10**6) #uH
actual_inductance = 2000*10**-6 #H
actual_ripple_current = ((max_pack_voltage-vout)*(vout/(max_pack_voltage)))/(fsw*actual_inductance)
print(actual_ripple_current)

cout = actual_ripple_current/(8*fsw*acceptable_dvout)
print(cout)
actual_cout = np.array([[2, 0.07, 70], #100nF
                        [4, 4.4, 2.6], #10uF
                        [2, 1.2, 2.5]]) #4.7u
actual_cout[:, 1] = actual_cout[:, 1]*10**-6 # Convert to F
actual_cout[:, 2] = actual_cout[:, 2]*10**-3 # Convert to Ohms
print(actual_cout)

capacitances = equiv_capacitance(actual_cout, fsw)
effective_capacitance = capacitances.equivalent_capacitance
effective_esr = capacitances.equivalent_esr
print(effective_capacitance)
print(effective_esr)

0.53125
5490.836334533814
1.4585034013605442
7.292517006802721e-07
[[2.0e+00 7.0e-08 7.0e-02]
 [4.0e+00 4.4e-06 2.6e-03]
 [2.0e+00 1.2e-06 2.5e-03]]
2.0139984732415844e-05
0.0005158279757812212


In [38]:
def gain_at_crossover(v_filter, v_ramp, C, Rc, L, Rl, rout, f_crossover):    
    # https://www.ti.com/lit/an/slva301/slva301.pdf?ts=1745660332504
    # Equation 17
    b1 = C*Rc*2*3.1416*f_crossover
    a1 = Rl/rout
    b2 = 2*3.1416*f_crossover*((L/rout)+Rc*C+Rl*C+((Rl*Rc*C)/rout))
    a2 = -(1 + Rc/rout)*L*C*(2*3.1416*f_crossover)**2

    rational = complex(1, b1)/complex(1 + a1 + a2, b2)
    return 1/((v_filter/v_ramp)*cmath.polar(rational)[0])


In [39]:
# Compensation Network Calculations
# https://www.ti.com/lit/an/slva301/slva301.pdf?ts=1745660332504

v_ramp = 2.4
vref = 2.5
rbot = 4700 #Ohms
f_crossover = fsw/5
inductor_dcr = 0.003 #ohms
rout = vout/iout #TODO check which current to use here and/or sweep

a_mod_ccm = (max_pack_voltage)/v_ramp
print(a_mod_ccm)
r1 = rbot*((vout/vref)-1)
print(f"R1 is {round(r1*10**-3,2)} kOhms")
f_lc = 1/(2*3.1416*math.sqrt(actual_inductance*effective_capacitance))
print(f"Filter xo is {round(f_lc*10**-3,2)} kHz")
c3 = 1/(2*3.1416*0.9*f_lc*r1)
print(f"C3 is {round(c3*10**12,2)} pF")
r3 = 1/(2*3.1416*f_crossover*c3)
print(f"R3 is {round(r3*10**-3,2)} kOhms")
crossover_gain = gain_at_crossover(float(max_pack_voltage), v_ramp, effective_capacitance, effective_esr, actual_inductance, inductor_dcr, rout, f_crossover)
print(crossover_gain)
r2 = parallel_resistance(r1, r3)*crossover_gain
print(f"R2 is {round(r2*10**-3,2)} kOhms")
c2 = 1/(1.8*3.1416*f_lc*r2)
print(f"C2 is {round(c2*10**9,2)} nF")
c1 = 1/(2*3.1416*10*f_crossover*r2)
print(f"C1 is {round(c1*10**12,2)} pF")

245.0
R1 is 596.9 kOhms
Filter xo is 0.79 kHz
C3 is 373.59 pF
R3 is 42.6 kOhms
0.6449950360212493
R2 is 25.65 kOhms
C2 is 8.69 nF
C1 is 62.06 pF


In [6]:
# Input Capacitor Sizing

C_in_ripple = (0.5*0.5*850/350)/(acceptable_dv_in*fsw)
print(C_in_ripple*10**6)

dv_step = 10
C_in_bulk = (0.5*(pmax/min_pack_voltage)*(min_pack_voltage/vout))/(dv_step*4*fsw/5)
print(C_in_bulk*10**6)

1.2142857142857142
3.3203125
